# HBFMID E1/E2 baseline (Gate A, Gate A2, noise floor)

| Run | What it is |
|---|---|
| **E1** | YOLOv8s on the official split + leak probe (48 test images with a train copy vs 16 without) |
| **E2** | YOLOv8s on HBFMID-Clean folds 0-4 (group-aware, no leakage) |
| **Noise floor** | extra seeds on clean fold 0, to measure seed-to-seed spread |

**Settings:** Accelerator = *GPU T4 x2* (not P100), Internet = ON.
**Order of use:** (1) leave `SMOKE = True` and *Run All* in an interactive session (about 10 min) to prove the pipeline works;
(2) set `SMOKE = False` and use *Save Version -> Save & Run All (Commit)*.
The dataset input is read-only, so Ultralytics cannot write a label cache into it (intended).


In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
os.environ["WANDB_MODE"] = "disabled"          # no experiment-tracker prompts

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 in Session options, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), "Dataset not attached or path differs: " + DATA_ROOT

SMOKE = True            # True = 2 epochs, official + clean fold 0 only (pipeline test). Set False for the real run.
MODEL = "yolov8s.pt"
USE_ALL_GPUS = True     # T4 x2: run two trainings at the same time (one per GPU)

if SMOKE:
    EPOCHS, OUT = 2, "/kaggle/working/results/smoke"
    PLAN = [("official", 0, 42, True), ("clean", 0, 42, False)]
else:
    EPOCHS, OUT = 100, "/kaggle/working/results/baseline"
    PLAN  = [("official", 0, 42, True)]                    # E1 (keeps best.pt)
    PLAN += [("clean", k, 42, False) for k in range(5)]    # E2, folds 0-4
    PLAN += [("clean", 0, s, False) for s in (43, 44)]     # noise floor
print(len(PLAN), "runs, epochs =", EPOCHS, "->", OUT)


In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count())) if USE_ALL_GPUS else [0]
q = queue.Queue()
for t in PLAN:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            mode, fold, seed, keep = q.get_nowait()
        except queue.Empty:
            return
        name = f"{mode}_f{fold}_s{seed}"
        cmd = [sys.executable, f"{REPO}/scripts/training/run_baseline.py",
               "--mode", mode, "--fold", str(fold), "--seed", str(seed), "--model", MODEL,
               "--epochs", str(EPOCHS), "--device", str(gpu), "--data-root", DATA_ROOT,
               "--splits-dir", f"{REPO}/data_splits", "--out", OUT]
        if keep:
            cmd.append("--keep-weights")
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:                                    # show why
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))


In [ ]:
import shutil
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_baseline.py", "--runs", OUT])
shutil.make_archive("/kaggle/working/baseline_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if SMOKE:
    print("\nSMOKE test done. Check the log tails above, then set SMOKE = False and use Save Version.")
if failed:
    print("\nWARNING: some runs failed:", failed, "- send me their logs from", OUT + "/logs")


After the real run: download `baseline_results.zip` from the version's Output tab, unzip it into `results/baseline/` in the local repo, and commit.
Pin the Ultralytics version printed in the setup cell in `requirements.txt`.
